# Hypothesis Testing – Vanguard A/B Test

This notebook evaluates whether the new digital design performs differently from the traditional design based on selected KPIs.

The analysis uses the pre-computed KPI tables created in the updated project structure:

- `clients_kpi_table.csv`
- `visits_kpi_table.csv`
- `events_kpi_table.csv`

For each KPI, we follow the same structure:

1. Define the KPI and business question
2. Choose the appropriate statistical test
3. Formulate null and alternative hypotheses
4. Run the test
5. Interpret the result

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np

from scipy.stats import ttest_ind, chi2_contingency

## Load KPI Tables

We load the pre-computed KPI tables created during the data preparation step.

Each table represents a different analysis level:

- Client level: one row per client
- Visit level: one row per visit
- Event level: one row per interaction/event

In [ ]:
# Load pre-computed KPI tables

df_clients = pd.read_csv("../data/clean/client_kpi_table.csv")
df_visits = pd.read_csv("../data/clean/visits_kpi_table.csv")
df_events = pd.read_csv("../data/clean/events_kpi_table.csv")

In [ ]:
df_clients.columns

In [ ]:
df_visits.columns

In [ ]:
df_events.columns

## KPI Overview

| KPI                          | Level  | Column / Logic                                      | Type       | Test       | Definition |
|------------------------------|--------|-----------------------------------------------------|------------|------------|------------|
| Frictionless Completion      | Client | frictionless (reached_confirm & no friction events) | Binary     | Chi-square | Proportion of clients with at least one frictionless completed visit |
| Time to Completion           | Client | avg_time_to_completion                              | Continuous | T-test     | Average time to complete the process per client |
| Time Spent per Step          | Event  | time_on_each_step                                   | Continuous | T-test     | Average time spent on each process step |
| Friction Rate (Error Rate)   | Visit  | went_backwards OR had_skip OR had_repeat            | Binary     | Chi-square | Proportion of visits with at least one friction event |
| Step Conversion Rate         | Event  | next_step progression                              | Binary     | Chi-square | Probability of progressing from one step to the next |
| Sessions per Client          | Client | sessions_per_client                                 | Continuous | T-test     | Number of sessions per client |
| Avg Steps per Session        | Client | avg_steps_per_session                               | Continuous | T-test     | Average number of steps per session |

## KPI 1: Frictionless Completion Rate

### Business Question
Does the new design increase the probability that a client completes the process without friction?

### Metric
Frictionless Completion Rate = proportion of clients who have at least one frictionless completed visit.

A frictionless visit is defined as a visit where:
- the client reaches the 'confirm' step
- the visit includes no backward steps, skips, or repeated steps

### Statistical Test
Chi-square test of independence

In [ ]:
# Define frictionless visits at visit level
df_visits["frictionless"] = (
    (df_visits["reached_confirm"] == 1) &
    (df_visits["went_backwards"] == 0) &
    (df_visits["had_skip"] == 0) &
    (df_visits["had_repeat"] == 0)
)

# Sanity check
df_visits["frictionless"].value_counts(normalize=True)

In [ ]:
# Aggregate to client level
# Did the client EVER have a frictionless completion?

user_frictionless = df_visits.groupby("client_id")["frictionless"].max()

# Convert to DataFrame
user_frictionless = user_frictionless.reset_index()

# Check
user_frictionless.head()

In [ ]:
# Sanity check (user level)
user_frictionless["frictionless"].value_counts(normalize=True)

In [ ]:
# Merge variation onto user-level dataset
user_frictionless = user_frictionless.merge(
    df_clients[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

# Check
user_frictionless.head()

In [ ]:
# Keep only users in experiment
user_frictionless = user_frictionless[user_frictionless["Variation"].notna()].copy()

# Check counts
user_frictionless["Variation"].value_counts()

In [ ]:
# Frictionless completion rate per group (USER level)
frictionless_rate = user_frictionless.groupby("Variation")["frictionless"].mean()

print(frictionless_rate)

In [ ]:
# Calculate relative lift in frictionless completion rate
control_rate = frictionless_rate.loc["Control"]
test_rate = frictionless_rate.loc["Test"]

lift = (test_rate - control_rate) / control_rate

print("Control frictionless completion rate:", control_rate)
print("Test frictionless completion rate:", test_rate)
print("Relative lift:", lift)

### Hypothesis

Let:

- p_control = probability that a client has at least one frictionless completion in the Control group  
- p_test = probability that a client has at least one frictionless completion in the Test group  

H0 (Null Hypothesis):  
p_test = p_control  

H1 (Alternative Hypothesis):  
p_test ≠ p_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(
    user_frictionless["Variation"],
    user_frictionless["frictionless"]
)

contingency

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(contingency)

print("Chi2:", chi2)
print("p-value:", p_value)

### Interpretation — Frictionless Completion Rate

The frictionless completion rate is higher in the Test group (43.9%) compared to the Control group (40.5%).

The Chi-square test shows that this difference is statistically significant (p < 0.05), indicating that the observed difference is unlikely to be due to random variation.

This suggests that the new design helps more users complete the process without friction (i.e., without errors or backward navigation).

However, the absolute improvement is moderate (~3.4 percentage points), corresponding to a relative lift of ~8.2%, which should be considered when evaluating the practical impact of the change.

## KPI 2: Time to Completion

### Hypothesis

Let:
- μ_test = average time to completion in the Test group
- μ_control = average time to completion in the Control group

**H0 (Null Hypothesis):**  
μ_test = μ_control  

**H1 (Alternative Hypothesis):**  
μ_test ≠ μ_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

In [ ]:
# Extract average time to completion per client by group
control_times = df_clients[df_clients["Variation"] == "Control"]["avg_time_to_completion"]
test_times = df_clients[df_clients["Variation"] == "Test"]["avg_time_to_completion"]

print("Control mean:", control_times.mean())
print("Test mean:", test_times.mean())

In [ ]:
from scipy.stats import ttest_ind

t_stat_time, p_value_time = ttest_ind(
    test_times.dropna(),
    control_times.dropna()
)

print("t-statistic:", t_stat_time)
print("p-value:", p_value_time)

### Interpretation – Time to Completion

The average time to completion is lower in the Test group (248.1 seconds) compared to the Control group (289.9 seconds).

The t-test shows that this difference is statistically significant (p < 0.05), indicating that the observed reduction in completion time is unlikely to be due to random variation.

This suggests that the new design allows users to complete the process faster.

The absolute improvement is approximately 41.8 seconds, which represents a meaningful reduction in user effort and time spent.

Overall, the new design improves efficiency by speeding up the completion process.

## KPI 3: Time Spent on Each Step

### Business Question
Does the new design change how much time users spend on each step?

### Metric
Average time spent per step (`time_on_each_step`) by process step.

### Statistical Test
Two-sample T-test (per step)

### Hypothesis – Time Spent on Step 1

We test whether the average time spent on **step_1** differs between the Test and Control groups.

Let:
- μ_test = average time on step_1 in the Test group  
- μ_control = average time on step_1 in the Control group  

**H0 (Null Hypothesis):**  
μ_test = μ_control  

**H1 (Alternative Hypothesis):**  
μ_test ≠ μ_control  

This is a two-sided test.

We use a two-sample T-test because `time_on_each_step` is a continuous variable and we compare two independent groups.

In [ ]:
# Add Variation (Test / Control) to event-level data
df_events_ab_test = df_events.merge(
    df_clients[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

# Keep only events from experiment clients
df_events_ab_test = df_events_ab_test[df_events_ab_test["Variation"].notna()].copy()

# Check
df_events_ab_test[["client_id", "visit_id", "process_step", "Variation", "time_on_each_step"]].head()

In [ ]:
from scipy.stats import ttest_ind

step1 = df_events_ab_test[df_events_ab_test["process_step"] == "step_1"]

control = step1[step1["Variation"] == "Control"]["time_on_each_step"]
test = step1[step1["Variation"] == "Test"]["time_on_each_step"]

t_stat_1, p_val_1 = ttest_ind(test.dropna(), control.dropna())

print("Step 1 - t-stat:", t_stat_1)
print("Step 1 - p-value:", p_val_1)

### Interpretation

The p-value is far below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the average time spent on step 1 between the Test and Control groups.

Since the t-statistic is negative, users in the Test group spend **less time** on step 1 compared to the Control group.

This suggests that the new design allows users to move through step 1 more efficiently.

In [ ]:
step2 = df_events_ab_test[df_events_ab_test["process_step"] == "step_2"]

control = step2[step2["Variation"] == "Control"]["time_on_each_step"]
test = step2[step2["Variation"] == "Test"]["time_on_each_step"]

t_stat_2, p_val_2 = ttest_ind(test.dropna(), control.dropna())

print("Step 2 – t-stat:", t_stat_2)
print("Step 2 – p-value:", p_val_2)

### Interpretation – Step 2

The p-value is far below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the average time spent on step 2 between the Test and Control groups.

Since the t-statistic is positive, users in the Test group spend more time on step 2 compared to the Control group.

This suggests that step 2 in the new design may be less efficient or introduce additional friction for users.

In [ ]:
step3 = df_events_ab_test[df_events_ab_test["process_step"] == "step_3"]

control = step3[step3["Variation"] == "Control"]["time_on_each_step"]
test = step3[step3["Variation"] == "Test"]["time_on_each_step"]

t_stat_3, p_val_3 = ttest_ind(test.dropna(), control.dropna())

print("Step 3 – t-stat:", t_stat_3)
print("Step 3 – p-value:", p_val_3)

### Interpretation – Step 3

The p-value is below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the average time spent on step 3 between the Test and Control groups.

Since the t-statistic is positive, users in the Test group spend more time on step 3 compared to the Control group.

This suggests that step 3 in the new design may be less efficient or require more time to complete.

In [ ]:
confirm = df_events_ab_test[df_events_ab_test["process_step"] == "confirm"]

control = confirm[confirm["Variation"] == "Control"]["time_on_each_step"]
test = confirm[confirm["Variation"] == "Test"]["time_on_each_step"]

t_stat_confirm, p_val_confirm = ttest_ind(test.dropna(), control.dropna())

print("Confirm – t-stat:", t_stat_confirm)
print("Confirm – p-value:", p_val_confirm)

### Interpretation – Confirm Step

The p-value is above the significance level (α = 0.05), therefore we fail to reject the null hypothesis.

There is no statistically significant difference in the average time spent on the confirm step between the Test and Control groups.

This suggests that the new design does not meaningfully change the time required to complete the final confirmation step.

## KPI 4: Friction Rate (Error Rate)

### Business Question
Does the new design affect how often users experience friction during a session?

### Metric Definition
Friction Rate is defined as the proportion of visits where at least one friction event occurs.

A friction event is defined as any of the following:
- Going backwards (`went_backwards`)
- Skipping steps (`had_skip`)
- Repeating steps (`had_repeat`)

### Statistical Test
Chi-square test of independence (binary outcome: friction vs no friction)

In [ ]:
# Create friction flag per visit

df_visits["friction"] = (
    (df_visits["went_backwards"] == 1) |
    (df_visits["had_skip"] == 1) |
    (df_visits["had_repeat"] == 1)
)

# Convert boolean to int (optional but cleaner)
df_visits["friction"] = df_visits["friction"].astype(int)

# Quick check
df_visits["friction"].value_counts()


In [ ]:
# Merge variation from clients
df_visits_ab = df_visits.merge(
    df_clients[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

# Keep only experiment users
df_visits_ab = df_visits_ab[df_visits_ab["Variation"].notna()].copy()

# Check
df_visits_ab["Variation"].value_counts()

In [ ]:
import pandas as pd

friction_table = pd.crosstab(
    df_visits_ab["Variation"],
    df_visits_ab["friction"]
)

friction_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(friction_table)

print("Chi2:", chi2)
print("p-value:", p_value)

In [ ]:
# Calculate friction rates
friction_rates = friction_table.div(friction_table.sum(axis=1), axis=0)

control_rate = friction_rates.loc["Control", 1]
test_rate = friction_rates.loc["Test", 1]

# Relative lift
lift = (test_rate - control_rate) / control_rate

print("Control friction rate:", control_rate)
print("Test friction rate:", test_rate)
print("Relative lift:", lift)

### Interpretation – Friction Rate

The p-value is extremely small (p < 0.001), which is far below the significance level of 0.05.

→ We reject the null hypothesis.

There is a statistically significant difference in the proportion of visits with friction between the Test and Control groups.

The Test group shows a higher friction rate (~0.46) compared to the Control group (~0.41), corresponding to a relative increase of about +11%.

This suggests that users in the Test group experience friction more frequently during their sessions.

From a business perspective, this indicates that the new design introduces additional usability issues, such as more backward navigation, repeated actions, or skipped steps.

Overall, while the new design may improve certain outcomes (e.g., conversion), it also increases friction along the user journey.

### Additional Insight

Friction is defined as any occurrence of backward navigation, skipped steps, or repeated actions, capturing multiple forms of user struggle within a session.

## KPI 5: Step Conversion Rate

### Business Question
Does the new design change how successfully users move forward through the process steps?

### Metric
Step Conversion Rate = probability of moving forward from one process step to the next.

For this analysis, we use event-level transition data and define a successful conversion as a forward movement:

- from `start` to `step_1`
- from `step_1` to `step_2`
- from `step_2` to `step_3`
- from `step_3` to `confirm`

### Statistical Test
Chi-square test of independence

### Hypothesis

Let:

- p_control = probability of moving forward to the next step in the Control group  
- p_test = probability of moving forward to the next step in the Test group  

H0 (Null Hypothesis):  
p_test = p_control  

H1 (Alternative Hypothesis):  
p_test ≠ p_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

We test this separately for each step transition.

In [ ]:
# Keep only events from clients who are part of the experiment
# (same clean logic as in previous KPIs)

df_events_ab = df_events.merge(
    df_clients[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

# Filter to only Test / Control users
df_events_ab = df_events_ab[df_events_ab["Variation"].notna()].copy()


# Sort events to ensure correct step order within each visit
df_events_ab = df_events_ab.sort_values(
    by=["client_id", "visit_id", "date_time"]
)


# Quick sanity checks

print(df_events_ab["Variation"].value_counts())
print(df_events_ab["process_step"].value_counts())

In [ ]:
# Create next step per event (within each visit)

df_events_ab["next_step"] = (
    df_events_ab
    .groupby(["client_id", "visit_id"])["process_step"]
    .shift(-1)
)

# Check
df_events_ab[["process_step", "next_step"]].head(10)

### Step Conversion: Start → Step 1

We analyze whether users in the Test group are more likely to progress 
from "start" to "step_1" compared to the Control group.

Let:

p_test = probability of progressing from start to step_1 in the Test group  
p_control = probability of progressing from start to step_1 in the Control group  

H0 (Null Hypothesis): p_test = p_control  
H1 (Alternative Hypothesis): p_test ≠ p_control  

We use a Chi-square test of independence to compare progression rates between groups.

In [ ]:
# Filter rows where the current step is "start"
start_events = df_events_ab[df_events_ab["process_step"] == "start"].copy()

# Create binary outcome: did the user go to step_1?
start_events["progressed"] = start_events["next_step"] == "step_1"

# Contingency table
start_table = pd.crosstab(
    start_events["Variation"],
    start_events["progressed"]
)

start_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(start_table)

print("Chi2:", chi2)
print("p-value:", p_value)

### Interpretation

The p-value is extremely small (p < 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the probability of progressing 
from "start" to "step_1" between the Test and Control groups.

Users in the Test group are more likely to proceed to the next step compared to users in the Control group.

This suggests that the new design improves early-stage conversion in the process.

### Conversion Rates & Lift

To better understand the magnitude of the difference, we calculate the conversion rates 
for each group and the relative lift of the Test group compared to the Control group.

In [ ]:
# Calculate conversion rates per group
conversion_rates = start_table.div(start_table.sum(axis=1), axis=0)

# Extract values
control_rate = conversion_rates.loc["Control", True]
test_rate = conversion_rates.loc["Test", True]

# Calculate relative lift
lift = (test_rate - control_rate) / control_rate

# Print results
print("Control conversion rate:", control_rate)
print("Test conversion rate:", test_rate)
print("Relative lift:", lift)

### Interpretation – Start → Step 1 Conversion

The conversion rate from "start" to "step_1" is higher in the Test group (59.1%) compared to the Control group (54.1%).

The Chi-square test shows that this difference is statistically significant (p < 0.05), so we reject the null hypothesis.

The relative lift is approximately 9.1%, meaning that users in the Test group are more likely to progress from the start of the process to step 1.

This suggests that the new design improves early-stage conversion.

### Step Conversion: Step 1 → Step 2

We analyze whether users in the Test group are more likely to progress 
from "step_1" to "step_2" compared to the Control group.

Let:

p_test = probability of progressing from step_1 to step_2 in the Test group  
p_control = probability of progressing from step_1 to step_2 in the Control group  

H0 (Null Hypothesis): p_test = p_control  
H1 (Alternative Hypothesis): p_test ≠ p_control  

We use a Chi-square test of independence to compare progression rates between groups.

In [ ]:
# Filter step_1 events
step1_events = df_events_ab[df_events_ab["process_step"] == "step_1"].copy()

# Did user go to step_2?
step1_events["progressed"] = step1_events["next_step"] == "step_2"

# Contingency table
step1_table = pd.crosstab(
    step1_events["Variation"],
    step1_events["progressed"]
)

step1_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(step1_table)

print("Chi2:", chi2)
print("p-value:", p_value)

In [ ]:
conversion_rates = step1_table.div(step1_table.sum(axis=1), axis=0)

control_rate = conversion_rates.loc["Control", True]
test_rate = conversion_rates.loc["Test", True]

lift = (test_rate - control_rate) / control_rate

print("Control conversion rate:", control_rate)
print("Test conversion rate:", test_rate)
print("Relative lift:", lift)

### Interpretation – Step 1 → Step 2

The conversion rate from "step_1" to "step_2" is slightly lower in the Test group (70.9%) compared to the Control group (75.3%).

The Chi-square test shows that this difference is statistically significant (p < 0.05), so we reject the null hypothesis.

The relative lift is approximately -5.7%, indicating that users in the Test group are less likely to progress from step 1 to step 2.

This suggests that the new design may introduce friction or usability issues at this stage of the process.

### Step Conversion: Step 2 → Step 3

We analyze whether users in the Test group are more likely to progress 
from "step_2" to "step_3" compared to the Control group.

Let:

p_test = probability of progressing from step_2 to step_3 in the Test group  
p_control = probability of progressing from step_2 to step_3 in the Control group  

H0 (Null Hypothesis): p_test = p_control  
H1 (Alternative Hypothesis): p_test ≠ p_control  

We use a Chi-square test of independence to compare progression rates between groups.

In [ ]:
# Filter step_2 events
step2_events = df_events_ab[df_events_ab["process_step"] == "step_2"].copy()

# Did user go to step_3?
step2_events["progressed"] = step2_events["next_step"] == "step_3"

# Contingency table
step2_table = pd.crosstab(
    step2_events["Variation"],
    step2_events["progressed"]
)

step2_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(step2_table)

print("Chi2:", chi2)
print("p-value:", p_value)

In [ ]:
conversion_rates = step2_table.div(step2_table.sum(axis=1), axis=0)

control_rate = conversion_rates.loc["Control", True]
test_rate = conversion_rates.loc["Test", True]

lift = (test_rate - control_rate) / control_rate

print("Control conversion rate:", control_rate)
print("Test conversion rate:", test_rate)
print("Relative lift:", lift)

### Interpretation

The p-value is far below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the probability of progressing from step_2 to step_3 between the Test and Control groups.

The conversion rate in the Test group (~0.78) is slightly lower than in the Control group (~0.82), resulting in a negative relative lift (~ -4.9%).

This suggests that the new design reduces progression efficiency at step 2, indicating potential friction or usability issues at this stage.

Step Conversion: Step 3 → Confirm (Chi-Square Test)

We analyze whether users in the Test group are more or less likely to progress from step_3 to confirm compared to the Control group.

Let:

* p_test = probability of progressing from step_3 to confirm in the Test group
* p_control = probability of progressing from step_3 to confirm in the Control group

H0 (Null Hypothesis):
p_test = p_control

H1 (Alternative Hypothesis):
p_test ≠ p_control

We use a Chi-square test of independence to compare progression rates between groups.

In [ ]:
# Filter step_3 events
step3_events = df_events_ab[df_events_ab["process_step"] == "step_3"].copy()

# Define progression to confirm
step3_events["progressed"] = step3_events["next_step"] == "confirm"

# Contingency table
step3_table = pd.crosstab(
    step3_events["Variation"],
    step3_events["progressed"]
)

step3_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(step3_table)

print("Chi2:", chi2)
print("p-value:", p_value)

In [ ]:
# Conversion rates & lift
conversion_rates = step3_table.div(step3_table.sum(axis=1), axis=0)

control_rate = conversion_rates.loc["Control", True]
test_rate = conversion_rates.loc["Test", True]

lift = (test_rate - control_rate) / control_rate

print("Control conversion rate:", control_rate)
print("Test conversion rate:", test_rate)
print("Relative lift:", lift)

### Interpretation

The p-value is far below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the probability of progressing from step_3 to confirm between the Test and Control groups.

The conversion rate in the Test group (~0.71) is higher than in the Control group (~0.67), resulting in a positive relative lift (~ +5.8%).

This suggests that the new design improves progression efficiency at step 3, helping more users successfully reach the final confirmation step.

### Overall Interpretation – Step Conversion Funnel

Across the funnel, the impact of the new design is mixed.

The Test group shows improved conversion in early (start → step_1) and late stages (step_3 → confirm), but a noticeable drop in progression at step_2.

This suggests that while the new design helps users enter and complete the process more efficiently, it introduces friction in the middle of the journey.

Overall, the results indicate a potential usability issue specifically at step_2, which may be limiting the full positive impact of the redesign.

Further investigation is recommended to understand what causes the drop-off at this stage.

## KPI 6: Sessions per Client

### Business Question  
Does the new design change how many sessions users need to complete the process?

---

### Hypothesis

Let:  
- μ_test = average number of sessions per client in the Test group  
- μ_control = average number of sessions per client in the Control group  

H0 (Null Hypothesis):  
μ_test = μ_control  

H1 (Alternative Hypothesis):  
μ_test ≠ μ_control  

This is a two-sided test.

We use a two-sample t-test because `session_per_client` is a continuous variable and we compare two independent groups.

In [ ]:
from scipy.stats import ttest_ind

# Extract sessions per client for each group
control_sessions = df_clients[df_clients["Variation"] == "Control"]["session_per_client"].dropna()
test_sessions = df_clients[df_clients["Variation"] == "Test"]["session_per_client"].dropna()

# Run independent t-test (Welch's t-test)
t_stat_sessions, p_value_sessions = ttest_ind(test_sessions, control_sessions, equal_var=False)

# Print results
print("Control mean:", control_sessions.mean())
print("Test mean:", test_sessions.mean())
print("t-statistic:", t_stat_sessions)
print("p-value:", p_value_sessions)

### Interpretation – Sessions per Client

The p-value (≈ 0.21) is greater than the significance level (α = 0.05), therefore we fail to reject the null hypothesis.

There is no statistically significant difference in the average number of sessions per client between the Test and Control groups.

Although the Test group shows a slightly higher mean (~1.38 vs ~1.37), this difference is very small and not statistically meaningful.

This suggests that the new design does not significantly impact how many sessions users need to complete the process.

## KPI 7: Average Steps per Client

### Business Question
Does the new design change the average number of steps users take per session?

### Metric
Average Steps per Session = average number of process steps taken within a user session.

This KPI helps us understand whether the new design changes the complexity or structure of the user journey.

### Statistical Test
Two-sample t-test

### Hypothesis

Let:

- μ_test = average number of steps per session in the Test group
- μ_control = average number of steps per session in the Control group

H0 (Null Hypothesis):  
μ_test = μ_control

H1 (Alternative Hypothesis):  
μ_test ≠ μ_control

This is a two-sided test, as we are interested in detecting any difference between the groups.

We use a two-sample t-test because `avg_steps_per_session` is a numerical variable and we compare the means of two independent groups.

In [ ]:
from scipy.stats import ttest_ind

# Extract average steps per client for each group
control_steps = df_clients[df_clients["Variation"] == "Control"]["avg_steps_client"].dropna()
test_steps = df_clients[df_clients["Variation"] == "Test"]["avg_steps_client"].dropna()

# Run independent t-test (Welch)
t_stat_steps, p_value_steps = ttest_ind(
    test_steps,
    control_steps,
    equal_var=False
)

# Print results
print("Control mean:", control_steps.mean())
print("Test mean:", test_steps.mean())
print("t-statistic:", t_stat_steps)
print("p-value:", p_value_steps)

### Interpretation – Average Steps per Client

The p-value is extremely small (p < 0.001), which is far below the significance level of 0.05.

→ We reject the null hypothesis.

There is a statistically significant difference in the average number of steps per client between the Test and Control groups.

The Test group has a higher average number of steps (~5.08) compared to the Control group (~4.70).

This suggests that users in the Test group require more steps to complete the process.

From a business perspective, this indicates that the new design introduces additional interactions or complexity, potentially increasing friction along the user journey.

## Final Hypothesis Testing Insights

Across all KPIs, we observe a consistent pattern when comparing the Test and Control groups.

### Key Results

**Frictionless Completion Rate**
- Significantly higher in the Test group
- More users complete the process without any friction events

**Time to Completion**
- Significantly higher in the Test group
- Users take longer to complete the process

**Time Spent per Step**
- Mixed results across steps
- Some steps take longer in the Test group, suggesting increased complexity

**Friction Rate**
- Significantly higher in the Test group
- Users experience more friction events (backtracking, skipping, repeating steps)

**Sessions per Client**
- No significant difference
- Users do not require more sessions to complete the process

**Average Steps per Session**
- Significantly higher in the Test group
- Users go through more steps within a session

---

### Overall Interpretation

The new design improves completion outcomes, increasing the likelihood that users successfully finish the process.

However, this improvement comes at the cost of reduced efficiency:
- Users take longer
- Users perform more steps
- Users experience more friction events

This suggests that the redesigned flow provides more guidance or recovery options, helping users reach completion even when encountering difficulties.

---

### Business Insight

The redesign appears to prioritize **effectiveness over efficiency**:

- ✅ Higher completion reliability (more users reach the goal)
- ❗ Higher interaction cost (time, steps, friction)

This indicates a clear trade-off:
- The process becomes more robust and forgiving
- But also more complex and effort-intensive

---

### Recommendation Direction

- Identify steps with increased friction (e.g. backtracking or long durations)
- Simplify or streamline these steps without removing helpful guidance
- Focus on reducing unnecessary interactions while preserving the improved completion rate

Further analysis (e.g. step-level deep dives or user segmentation) is recommended to pinpoint optimization opportunities.